<a href="https://colab.research.google.com/github/DevzsJhonny/Challenge-AI-ONE---Agente-LOGICAR/blob/main/Desenv_Agente_LOGICAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Instalação das bibliotecas
!pip install langchain pypdf sentence-transformers chromadb langchain-community

#!pip install google-generativeai PyPDF2 python-dotenv

from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Carregar o PDF
loader = PyPDFLoader("dados_logicar.pdf")
documentos = loader.load()


text_splitter = RecursiveCharacterTextSplitter (
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documentos)

print(f"Documento carregado com sucesso!")
print(f"Total de pedaços: {len(chunks)}")

Documento carregado com sucesso!
Total de pedaços: 2


In [8]:
#!pip install -U langchain-google-genai chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 24.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [5]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA

minha_api_key = GEMINI_API_KEY

# 1. Configurar os Embeddings (essencial para o RAG encontrar os textos no PDF)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=minha_api_key
)

# 2. Criar o Banco de Dados com os 'chunks' do seu PDF da Soluções Logicar
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# 3. Configurar o LLM usando o modelo que funcionou no seu teste!
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=minha_api_key,
    temperature=0
)

# 4. Criar o Agente de Respostas
agente_logicar = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

print("✅ Agente Soluções Logicar ONLINE com o modelo Flash-Latest!")

✅ Agente Soluções Logicar ONLINE com o modelo Flash-Latest!


#Testando o Agente

In [7]:
pergunta = "De acordo com o manual, o que acontece se uma entrega de logística atrasar?"
try:
    resultado = agente_logicar.invoke(pergunta)
    print(f"RESPOSTA DO AGENTE: {resultado['result']}")
except Exception as e:
    print(f"Erro no teste: {e}")

RESPOSTA DO AGENTE: De acordo com o manual, caso uma entrega atrase mais de 4 horas além do prazo previsto, o cliente recebe 10% de desconto no próximo frete.


In [14]:
# criar template
from  langchain_core.prompts import ChatPromptTemplate, PromptTemplate

In [15]:
template_logicar = """
Você é o assistente virtual oficial da Soluções Logicar.
Sua missão é ajudar clientes e colaboradores com informações sobre logística e aluguel de carros.

Use APENAS os trechos do documento fornecidos abaixo para responder à pergunta.
Se a informação não estiver no texto, responda educadamente: "Sinto muito, mas não encontrei essa informação no manual da Soluções Logicar."

REGRAS DE RESPOSTA:
1. Seja conciso, claro e objetivo.
2. Use trechos do documento para validar sua resposta sempre que possível.
3. Não invente informações fora do contexto fornecido.
4. Mantenha sempre um tom profissional e prestativo.

CONTEXTO (Manuais da empresa):
{context}

PERGUNTA DO USUÁRIO:
{question}

RESPOSTA DO ASSISTENTE:"""

PROMPT = PromptTemplate(
    template=template_logicar,
    input_variables=["context", "question"]
)

In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA

minha_api_key = GEMINI_API_KEY

# 1. Configurar os Embeddings (essencial para o RAG encontrar os textos no PDF)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=minha_api_key
)

# 2. Criar o Banco de Dados com os 'chunks' do seu PDF da Soluções Logicar
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# 3. Configurar o LLM usando o modelo que funcionou no seu teste!
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=minha_api_key,
    temperature=0
)

# 4. Criar o Agente de Respostas (agora com template)
agente_logicar = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
    # AQUI entra o seu prompt personalizado:
    chain_type_kwargs={"prompt": PROMPT}
)

print("✅ Agente Soluções Logicar ONLINE com o modelo Flash-Latest!")




✅ Agente Soluções Logicar ONLINE com o modelo Flash-Latest!


In [17]:
# testando...
print("🤖 Assistente Soluções Logicar Inicializado!")
print("Digite sua pergunta abaixo (ou digite 'sair' para encerrar):")

while True:
    # Cria a caixa de texto para você digitar
    pergunta_usuario = input("\nVocê: ")

    # Condição para parar o loop
    if pergunta_usuario.lower() in ["sair", "parar", "exit", "quit"]:
        print("Encerrando o chat. Até logo!")
        break

    try:
        # O Agente processa a pergunta usando o RAG + o seu Prompt novo
        print("🔍 Consultando manual...")
        resultado = agente_logicar.invoke(pergunta_usuario)

        # Exibe a resposta final
        print("-" * 50)
        print(f"Agente Logicar: {resultado['result']}")
        print("-" * 50)

    except Exception as e:
        print(f"❌ Ocorreu um erro: {e}")

🤖 Assistente Soluções Logicar Inicializado!
Digite sua pergunta abaixo (ou digite 'sair' para encerrar):

Você: qual carro é o mais econômico?
🔍 Consultando manual...
--------------------------------------------------
Agente Logicar: O carro do Modelo Econômico disponível em nossa frota é o **Hyundai HB20 (2023)**, pelo valor de **R$ 120,00/diária**.
--------------------------------------------------

Você: como faço pra virar uber?
🔍 Consultando manual...
--------------------------------------------------
Agente Logicar: Sinto muito, mas não encontrei essa informação no manual da Soluções Logicar.
--------------------------------------------------


KeyboardInterrupt: Interrupted by user